# Hirriririir Multimodal Thigh Segmentation — Asian MRI Dataset (Water, Lambda)

Runs the **SegResNetDS** model from [Hirriririir/Multimodal-Multiethnic-Thigh-Muscle-MRI-analysis](https://github.com/Hirriririir/Multimodal-Multiethnic-Thigh-Muscle-MRI-analysis) on the `MRI_data_asian` Dixon WATER stacks — **Thigh only**.

The pretrained checkpoint (~330 MB) is downloaded automatically on first run.

Structure: `MRI_data_asian/MRI_data/{01-25}/Thigh/Water.nii.gz`  
Output: `~/hirriririir_asian_water_segs/{subject}/Thigh/Water_thigh_seg.nii.gz` (25 files)

## 1 — Upload data to Lambda
```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/MRI_data_asian \
  ubuntu@<YOUR-LAMBDA-IP>:~/
```

## 2 — Download results when done
```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  ubuntu@<YOUR-LAMBDA-IP>:~/hirriririir_asian_water_segs/ \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/multimodal-multiethnic/asian_segs_water/
```

**Terminate the instance when done.**

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'monai', 'SimpleITK'])
print('Dependencies ready')

In [ ]:
import glob, os
import numpy as np
import torch
import SimpleITK as sitk
from monai.networks.nets import SegResNetDS
from monai.inferers import sliding_window_inference

DATA_ROOT  = os.path.expanduser('~/MRI_data_asian/MRI_data')
OUTPUT_DIR = os.path.expanduser('~/hirriririir_asian_water_segs')
CHECKPOINT = os.path.expanduser('~/pretrained_segmentation_muscle.pt')

TARGET_SPACING = (0.7813, 0.7813, 4.0)
ROI_SIZE       = [336, 336, 88]
DEVICE         = 'cuda' if torch.cuda.is_available() else 'cpu'

LABEL_MAP = {
    1:  'Sartorius',
    2:  'Rectus_Femoris',
    3:  'Vastus_Lateralis',
    4:  'Vastus_Intermedius',
    5:  'Vastus_Medialis',
    6:  'Adductor_Magnus',
    7:  'Gracilis',
    8:  'Biceps_Femoris_Long',
    9:  'Semitendinosus',
    10: 'Semimembranosus',
    11: 'Biceps_Femoris_Short',
}

# Discover Thigh jobs
jobs = []
for subject in sorted(os.listdir(DATA_ROOT)):
    water_path = os.path.join(DATA_ROOT, subject, 'Thigh', 'Water.nii.gz')
    if os.path.exists(water_path):
        jobs.append((subject, water_path))

print(f'Device : {DEVICE}')
print(f'Found  : {len(jobs)} Thigh Water stacks')
for subj, p in jobs:
    print(f'  {subj}  →  {p}')

In [ ]:
# ── Download pretrained checkpoint (~330 MB) ──────────────────────────────────
if not os.path.exists(CHECKPOINT):
    import urllib.request
    print('Downloading checkpoint...')
    url = ('https://github.com/Hirriririir/Multimodal-Multiethnic-Thigh-Muscle-MRI-analysis'
           '/releases/download/1.0/pretrained_segmentation_muscle.pt')
    urllib.request.urlretrieve(url, CHECKPOINT)
    print(f'Done ({os.path.getsize(CHECKPOINT) // 1_000_000} MB)')
else:
    print('Checkpoint already present')

In [ ]:
# ── Load model ────────────────────────────────────────────────────────────────
model = SegResNetDS(
    spatial_dims=3,
    in_channels=1,
    out_channels=12,
    init_filters=32,
    blocks_down=(1, 2, 2, 4, 4),
    dsdepth=4,
    norm='INSTANCE',
    resolution=TARGET_SPACING,
)
ckpt  = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
state = (ckpt.get('state_dict') or ckpt.get('network_weights') or ckpt
         if isinstance(ckpt, dict) else ckpt)
missing, unexpected = model.load_state_dict(state, strict=False)
if missing:    print('Missing:   ', missing[:5])
if unexpected: print('Unexpected:', unexpected[:5])
model = model.to(DEVICE).eval()
print('Model ready on', DEVICE)

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────

def resample_sitk(sitk_img, new_spacing, interpolator=sitk.sitkLinear):
    orig_spacing = sitk_img.GetSpacing()
    orig_size    = sitk_img.GetSize()
    new_size = [
        int(round(orig_size[i] * orig_spacing[i] / new_spacing[i]))
        for i in range(3)
    ]
    r = sitk.ResampleImageFilter()
    r.SetOutputSpacing(new_spacing)
    r.SetSize(new_size)
    r.SetOutputDirection(sitk_img.GetDirection())
    r.SetOutputOrigin(sitk_img.GetOrigin())
    r.SetTransform(sitk.Transform())
    r.SetDefaultPixelValue(0)
    r.SetInterpolator(interpolator)
    return r.Execute(sitk_img)


def preprocess(nii_path):
    img  = sitk.ReadImage(nii_path, sitk.sitkFloat32)
    res  = resample_sitk(img, TARGET_SPACING)
    arr  = sitk.GetArrayFromImage(res).astype(np.float32).transpose(2, 1, 0)
    mask = arr > 0
    if mask.any():
        arr[mask] = (arr[mask] - arr[mask].mean()) / (arr[mask].std() + 1e-8)
    return arr, res, img


def infer_volume(arr):
    t = torch.tensor(arr[None, None]).float().to(DEVICE)
    with torch.no_grad():
        out = sliding_window_inference(
            t, roi_size=ROI_SIZE, sw_batch_size=1,
            predictor=model, overlap=0.5, mode='gaussian'
        )
    logits = out[0] if isinstance(out, (list, tuple)) else out
    return torch.argmax(logits, dim=1).squeeze(0).cpu().numpy().astype(np.uint8)


print('Helpers defined.')

In [ ]:
# ── Run segmentation ──────────────────────────────────────────────────────────

for subject, water_path in jobs:
    out_subdir = os.path.join(OUTPUT_DIR, subject, 'Thigh')
    out_path   = os.path.join(out_subdir, 'Water_thigh_seg.nii.gz')

    if os.path.exists(out_path):
        print(f'Skipping (done): {subject}/Thigh')
        continue

    print(f'\nProcessing: {subject}/Thigh')
    os.makedirs(out_subdir, exist_ok=True)

    arr, res_ref, orig = preprocess(water_path)
    print(f'  Array shape (nx, ny, nz): {arr.shape}')

    pred = infer_volume(arr)
    print(f'  Labels present: {sorted(np.unique(pred).tolist())}')

    # Transpose back from MONAI (nx, ny, nz) to SimpleITK (nz, ny, nx)
    pred_sitk = sitk.GetImageFromArray(pred.transpose(2, 1, 0))
    pred_sitk.CopyInformation(res_ref)
    pred_orig = sitk.Resample(pred_sitk, orig,
                               sitk.Transform(), sitk.sitkNearestNeighbor, 0)
    sitk.WriteImage(pred_orig, out_path)
    print(f'  Saved → {out_path}')

    pred_arr = sitk.GetArrayFromImage(pred_orig)
    print(f'  {"Label":<6} {"Muscle":<25} {"Voxels":>10}')
    for idx, name in LABEL_MAP.items():
        n = int((pred_arr == idx).sum())
        if n > 0:
            print(f'  {idx:<6} {name:<25} {n:>10,}')

print('\nAll done.')

In [ ]:
# ── Sanity check ─────────────────────────────────────────────────────────────
results = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*', 'Thigh', 'Water_thigh_seg.nii.gz')))
print(f'Output files found: {len(results)} / {len(jobs)}')
if results:
    arr = sitk.GetArrayFromImage(sitk.ReadImage(results[0]))
    print(f'Sample : {results[0]}')
    print(f'Shape  : {arr.shape}')
    print(f'Labels : {sorted(np.unique(arr[arr > 0]).tolist())}')